In [2]:
import os
import pandas as pd

# Load dataset
import kagglehub
path = kagglehub.dataset_download("gpreda/bbc-news")

# List files in path
print("files in dataset: ", os.listdir(path))

file_path = os.path.join(path,'bbc_news.csv')
df = pd.read_csv(file_path)

# Preview data
df.head(10)



Using Colab cache for faster access to the 'bbc-news' dataset.
files in dataset:  ['bbc_news.csv']


,title,pubDate,guid,link,description
0,Ukraine: Angry Zelensky vows to punish Russian...,"Mon, 07 Mar 2022 08:01:56 GMT",https://www.bbc.co.uk/news/world-europe-60638042,https://www.bbc.co.uk/news/world-europe-606380...,The Ukrainian president says the country will ...
1,War in Ukraine: Taking cover in a town under a...,"Sun, 06 Mar 2022 22:49:58 GMT",https://www.bbc.co.uk/news/world-europe-60641873,https://www.bbc.co.uk/news/world-europe-606418...,"Jeremy Bowen was on the frontline in Irpin, as..."
2,Ukraine war 'catastrophic for global food',"Mon, 07 Mar 2022 00:14:42 GMT",https://www.bbc.co.uk/news/business-60623941,https://www.bbc.co.uk/news/business-60623941?a...,One of the world's biggest fertiliser firms sa...
3,Manchester Arena bombing: Saffie Roussos's par...,"Mon, 07 Mar 2022 00:05:40 GMT",https://www.bbc.co.uk/news/uk-60579079,https://www.bbc.co.uk/news/uk-60579079?at_medi...,The parents of the Manchester Arena bombing's ...
4,Ukraine conflict: Oil price soars to highest l...,"Mon, 07 Mar 2022 08:15:53 GMT",https://www.bbc.co.uk/news/business-60642786,https://www.bbc.co.uk/news/business-60642786?a...,Consumers are feeling the impact of higher ene...
5,Ukraine war: PM to hold talks with world leade...,"Mon, 07 Mar 2022 08:33:29 GMT",https://www.bbc.co.uk/news/uk-60642926,https://www.bbc.co.uk/news/uk-60642926?at_medi...,Boris Johnson is to meet the Canadian and Dutc...
6,Ukraine war: UK grants 50 Ukrainian refugee vi...,"Mon, 07 Mar 2022 08:09:21 GMT",https://www.bbc.co.uk/news/uk-60640460,https://www.bbc.co.uk/news/uk-60640460?at_medi...,"The home secretary says she is ""surging capaci..."
7,TikTok limits services as Netflix pulls out of...,"Mon, 07 Mar 2022 00:11:59 GMT",https://www.bbc.co.uk/news/business-60641988,https://www.bbc.co.uk/news/business-60641988?a...,TikTok suspends live streaming and new content...
8,"Covid: Fourth jab for Scotland's vulnerable, a...","Mon, 07 Mar 2022 07:46:30 GMT",https://www.bbc.co.uk/news/uk-60640975,https://www.bbc.co.uk/news/uk-60640975?at_medi...,Five things you need to know about the coronav...
9,Protests across Russia see thousands detained,"Sun, 06 Mar 2022 23:23:59 GMT",https://www.bbc.co.uk/news/world-europe-60640204,https://www.bbc.co.uk/news/world-europe-606402...,"People have been held in 53 cities, from St Pe..."


In [3]:
import spacy

# Load the small english model
nlp = spacy.load("en_core_web_sm")

def preprocess_for_lda(text):
    # Process text through spacy
    doc = nlp(text)

    # Keep only words that are:
    # 1. Not stop words
    # 2. Not punctuation
    # 3. Are Nouns, Adjectives, or Verbs (most meaningful for topics)
    tokens = [token.lemma_.lower() for token in doc
              if not token.is_stop and not token.is_punct and token.pos_ in ['NOUN', 'ADJ', 'VERB']]

    return tokens

# Combine title and description for better results
df['full_text'] = df['title'] + " " + df['description']

# Apply the cleaning (this might take a minute)
df['cleaned_tokens'] = df['full_text'].apply(preprocess_for_lda)

# Preview the first few rows
print(df[['title', 'cleaned_tokens']].head())

                                               title  \
0  Ukraine: Angry Zelensky vows to punish Russian...   
1  War in Ukraine: Taking cover in a town under a...   
2         Ukraine war 'catastrophic for global food'   
3  Manchester Arena bombing: Saffie Roussos's par...   
4  Ukraine conflict: Oil price soars to highest l...   

                                      cleaned_tokens  
0  [angry, vow, punish, russian, atrocity, ukrain...  
1  [take, cover, town, attack, frontline, residen...  
2  [war, catastrophic, global, food, world, big, ...  
3  [bombing, parent, hear, truth, parent, bombing...  
4  [conflict, oil, price, soar, high, level, cons...  


In [5]:
!pip install gensim
import gensim
from gensim import corpora

# Create a dictionary (mapping every unique word to an ID)
dictionary = corpora.Dictionary(df['cleaned_tokens'])

# Filter out very rare and very common words to sharpen the discovery
dictionary.filter_extremes(no_below=5, no_above=0.5)

# Create the Corpus (Bag of Words representation)
corpus = [dictionary.doc2bow(text) for text in df['cleaned_tokens']]

# The DISCOVERY step: Train the LDA model
# We are asking the model to find 5 hidden topics
lda_model = gensim.models.LdaModel(corpus=corpus,
                                   id2word=dictionary,
                                   num_topics=5,
                                   random_state=100,
                                   passes=10)

# Show the discovered topics
for idx, topic in lda_model.print_topics(-1):
    print(f"Topic {idx}: {topic}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 62.2 MB/s eta 0:00:00
Topic 0: 0.019*"year" + 0.018*"say" + 0.016*"woman" + 0.015*"find" + 0.015*"police" + 0.013*"man" + 0.012*"death" + 0.010*"old" + 0.009*"murder" + 0.009*"tell"
Topic 1: 0.029*"say" + 0.020*"election" + 0.013*"party" + 0.012*"new" + 0.010*"government" + 0.010*"leader" + 0.009*"minister" + 0.009*"day" + 0.008*"change" + 0.008*"vote"
Topic 2: 0.035*"win" + 0.018*"final" + 0.012*"beat" + 0.010*"say" + 0.010*"world" + 0.010*"team" + 0.010*"star" + 0.009*"record" + 0.009*"year" + 0.009*"time"
Topic 3: 0.016*"say" + 0.011*"rise" + 0.010*"people" + 0.008*"warn" + 0.008*"new" + 0.008*"cut" + 0.007*"pay" + 0.007*"year" + 0.006*"help" + 0.006*"tax"
Topic 4: 0.024*"kill" + 0.021*"strike" + 0.019*"die" + 0.016*"attack" + 0.012*"israeli" + 0.011*"day" + 0.010*"people" + 0.009*"hit" + 0.009*"car" + 0.009*"city"


In [6]:
# Extract and display significant words
# num_words=10 gives us the top 10 most influential words for each topic
top_topics = lda_model.show_topics(num_topics=5, num_words=10, formatted=False)

for topic_id, words in top_topics:
    # Extract just the word strings for a cleaner look
    word_list = [word for word, weight in words]
    print(f"Topic {topic_id}: {', '.join(word_list)}")

    # Simple comment: show weights for the top 3 words to see their dominance
    print(f"   Sample weights: {words[0][0]} ({words[0][1]:.3f}), {words[1][0]} ({words[1][1]:.3f})")
    print("-" * 20)

Topic 0: year, say, woman, find, police, man, death, old, murder, tell
   Sample weights: year (0.019), say (0.018)
--------------------
Topic 1: say, election, party, new, government, leader, minister, day, change, vote
   Sample weights: say (0.029), election (0.020)
--------------------
Topic 2: win, final, beat, say, world, team, star, record, year, time
   Sample weights: win (0.035), final (0.018)
--------------------
Topic 3: say, rise, people, warn, new, cut, pay, year, help, tax
   Sample weights: say (0.016), rise (0.011)
--------------------
Topic 4: kill, strike, die, attack, israeli, day, people, hit, car, city
   Sample weights: kill (0.024), strike (0.021)
--------------------


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

# Prepare text for Scikit-learn (needs strings, not lists of tokens)
texts = [" ".join(tokens) for tokens in df['cleaned_tokens']]

# Create TF-IDF Matrix
# Tfidf with better filtering
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.90,          # Ignore words that appear in > 90% of docs
    min_df=5,             # Must appear in at least 5 docs
    stop_words='english',
    token_pattern=r'[a-zA-Z]{3,}' # ONLY keep words with letters, at least 3 chars long
)

tfidf = tfidf_vectorizer.fit_transform(texts)

# NMF
nmf_model = NMF(n_components=5, random_state=1).fit(tfidf)

# Show results
feature_names = tfidf_vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(nmf_model.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
    print(f"NMF Topic {topic_idx}: {', '.join(top_words)}")

NMF Topic 0: say, election, war, new, leader, minister, party, people, government, russian
NMF Topic 1: win, final, beat, reach, title, semi, score, victory, quarter, highlight
NMF Topic 2: year, old, die, police, man, murder, kill, woman, attack, death
NMF Topic 3: day, strike, week, past, pay, picture, selection, image, quiz, attention
NMF Topic 4: rise, rate, price, cost, energy, high, inflation, living, fall, mortgage


In [10]:
!pip install pyLDAvis
import pyLDAvis
import pyLDAvis.gensim_models

# Prepare the visualization
# Note: This uses the LDA model, corpus, and dictionary from our previous steps
vis_data = pyLDAvis.gensim_models.prepare(lda_model, corpus, dictionary)

# Display in a Jupyter Notebook or Google Colab
pyLDAvis.display(vis_data)

# Or save as an HTML file to open in a browser
# pyLDAvis.save_html(vis_data, 'lda_visualization.html')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 21.6 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
